In [2]:
import os
import requests
import tempfile
import zipfile
from pathlib import Path
import concurrent.futures

In [3]:
# PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))
PROJECT_ROOT = Path('../../')
DIR_BASE = PROJECT_ROOT / 'data/input/weather/'

In [4]:
years = list(range(2015, 2024))
months = [f"{i:02}" for i in range(1, 13)]

In [5]:
URL_BASE = "https://danepubliczne.imgw.pl/pl/datastore/getfiledown/Arch/Telemetria/Meteo/"
URL_DATA = URL_BASE + "{year}/Meteo_{year}-{month}.zip"

FILES_META = [
    "kody_parametr.csv",
    "kody_stacji.csv",
    "opis.txt"
]

In [6]:
def download_and_extract(url, extract_dir):
    os.makedirs(extract_dir, exist_ok=True)

    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Failed to download the file. Status code: {response.status_code}")
        return

    with tempfile.NamedTemporaryFile(delete=False) as temp_file:
        temp_file.write(response.content)
        temp_file_path = temp_file.name

    with zipfile.ZipFile(temp_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    os.unlink(temp_file_path)
    print(f"Downloaded file {url.split('/')[-1]} and extracted to {extract_dir}")


In [ ]:
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    for year in range(2015, 2024):
        for month in months:
            url = URL_DATA.format(year=year, month=month)
            dir = DIR_BASE / str(year) / month

            # Fix for 2021
            if year == 2021 and month in ['01', '02']:
                url = url.replace("zip", "ZIP")

            executor.submit(download_and_extract, url, dir)

Downloaded file Meteo_2021-05.zip and extracted to ../../data/input/weather/2021/05
Downloaded file Meteo_2021-01.ZIP and extracted to ../../data/input/weather/2021/01
Downloaded file Meteo_2021-02.ZIP and extracted to ../../data/input/weather/2021/02
Downloaded file Meteo_2021-04.zip and extracted to ../../data/input/weather/2021/04
Downloaded file Meteo_2021-06.zip and extracted to ../../data/input/weather/2021/06
Downloaded file Meteo_2021-03.zip and extracted to ../../data/input/weather/2021/03
Downloaded file Meteo_2021-08.zip and extracted to ../../data/input/weather/2021/08
Downloaded file Meteo_2021-07.zip and extracted to ../../data/input/weather/2021/07
Downloaded file Meteo_2021-09.zip and extracted to ../../data/input/weather/2021/09
Downloaded file Meteo_2021-10.zip and extracted to ../../data/input/weather/2021/10
Downloaded file Meteo_2021-11.zip and extracted to ../../data/input/weather/2021/11
Downloaded file Meteo_2021-12.zip and extracted to ../../data/input/weather/

In [6]:
for filename in FILES_META:
    url = URL_BASE + filename
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Failed to download the file. Status code: {response.status_code}")
        continue
    
    with open(DIR_BASE / filename, 'wb') as f:
        f.write(response.content) 
    
    print(f"Downloaded file {filename}")

Downloaded file kody_parametr.csv
Downloaded file kody_stacji.csv
Downloaded file opis.txt
